Dependancies Installation

In [1]:
!pip install -q transformers peft trl accelerate datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00


Model Processing

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# sanity check it generates before you touch LoRA at all
inputs = tokenizer("Explain what a for loop does.", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**inputs, max_new_tokens=50)[0]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Explain what a for loop does. A for loop is a control flow statement in programming that allows you to iterate over a sequence of values and perform some action on each value. It is commonly used when you need to execute a block of code repeatedly until all the elements in a sequence have


Pipeline Test

In [3]:
from datasets import load_dataset

# stand-in only, swap for your real programming.jsonl later
placeholder = load_dataset("tatsu-lab/alpaca", split="train[:300]")

def format_example(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"}

placeholder = placeholder.map(format_example)

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [4]:
!pip uninstall -y torchao -q

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # sanity check — should be a small %

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [6]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./adapter_test",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=placeholder,
    args=training_args,
)
trainer.train()

KeyboardInterrupt: 

In [ ]:
model.save_pretrained("drive/MyDrive/LoRA-Adapters/adapters/test_adapter")

inputs = tokenizer("Explain what a for loop does.", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**inputs, max_new_tokens=50)[0]))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


Explain what a for loop does. A、 《国网办事处
    
 
    
   

    /**
                              �、“____ ____ code, the total size =  �


In [ ]:
model.eval()

test_prompt = "### Instruction:\nExplain what a for loop does.\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)

print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
Explain what a for loop does.

### Response:
A for loop is used to repeat a block of code multiple times. It allows you to execute the same set of instructions repeatedly until a certain condition is met. The loop can be nested, meaning that it can be executed inside another loop or conditional statement.


Programming Adapter

In [ ]:
del model
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
from datasets import load_dataset

programming_data = load_dataset("json", data_files="drive/MyDrive/LoRA-Adapters/data/programming.jsonl", split="train")

def format_example(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}"}

programming_data = programming_data.map(format_example)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
model = get_peft_model(model, lora_config)  # reuse the same lora_config object from before

training_args = SFTConfig(
    output_dir="./programming_adapter_run",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
)

trainer = SFTTrainer(model=model, train_dataset=programming_data, args=training_args)
trainer.train()

model.save_pretrained("drive/MyDrive/LoRA-Adapters/adapters/programming_adapter")

Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,0.810219
20,0.790599
30,0.734656
40,0.743168
50,0.699185
60,0.714268
70,0.699451
80,0.687005
90,0.712962
100,0.682398


In [ ]:
model.eval()
test_prompt = "### Instruction:\nWhat is the difference between a list and a tuple in Python?\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
What is the difference between a list and a tuple in Python?

### Response:
In Python, both lists and tuples are used to store collections of items. However, there are some key differences between them:

1. Syntax: Lists are defined using square brackets [], while tuples are defined using parentheses (). This means that you can use either syntax for creating a list or a tuple.

2. Type: A list is an ordered collection of elements, whereas a tuple is also an ordered collection


In [ ]:
# with the Programming adapter (should already match what you saw)
model.eval()
test_prompt = "### Instruction:\nWhat is the difference between a list and a tuple in Python?\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    with_adapter = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print("WITH adapter:\n", tokenizer.decode(with_adapter[0], skip_special_tokens=True))

# same prompt, adapter disabled — pure base model
with torch.no_grad():
    with model.disable_adapter():
        without_adapter = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print("\nWITHOUT adapter (base model):\n", tokenizer.decode(without_adapter[0], skip_special_tokens=True))

WITH adapter:
 ### Instruction:
What is the difference between a list and a tuple in Python?

### Response:
In Python, both lists and tuples are used to store collections of items. However, there are some key differences between them:

1. Syntax: Lists are defined using square brackets [], while tuples are defined using parentheses (). This means that you can use either syntax for creating a list or a tuple.

2. Type: A list is an ordered collection of elements, whereas a tuple is also an ordered collection

WITHOUT adapter (base model):
 ### Instruction:
What is the difference between a list and a tuple in Python?

### Response:
A list is an ordered collection of items that can be accessed using indexing, while a tuple is an immutable sequence of items. In other words, a list allows you to add or remove elements at any position within its bounds, whereas tuples cannot. Lists are mutable, meaning their contents can be changed after they are created, but tuples are immutable, meaning th

Finance Adapter

In [ ]:
del model
torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
from datasets import load_dataset

finance_data = load_dataset("json", data_files="drive/MyDrive/LoRA-Adapters/data/finance.jsonl", split="train")

def format_example(ex):
    return {"text": f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}"}

finance_data = finance_data.map(format_example)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
model = get_peft_model(model, lora_config)

training_args = SFTConfig(
    output_dir="./finance_adapter_run",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
)

trainer = SFTTrainer(model=model, train_dataset=finance_data, args=training_args)
trainer.train()

model.save_pretrained("drive/MyDrive/LoRA-Adapters/adapters/finance_adapter")

Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,3.209959
20,2.988972
30,2.771486
40,2.750858
50,2.551001
60,2.554920
70,2.450492
80,2.483395
90,2.506108
100,2.413712


In [ ]:
model.eval()
test_prompt = "### Instruction:\nWhat is the difference between a stock and a bond?\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
What is the difference between a stock and a bond?

### Response:
A stock represents ownership in an entity, while a bond represents debt owed by that entity to its creditors.


In [ ]:
model.eval()
test_prompt = "### Instruction:\nWhat is the difference between a stock and a bond?\n\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    with_adapter = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print("WITH adapter:\n", tokenizer.decode(with_adapter[0], skip_special_tokens=True))

# same prompt, adapter disabled — pure base model
with torch.no_grad():
    with model.disable_adapter():
        without_adapter = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print("\nWITHOUT adapter (base model):\n", tokenizer.decode(without_adapter[0], skip_special_tokens=True))

WITH adapter:
 ### Instruction:
What is the difference between a stock and a bond?

### Response:
A stock represents ownership in an entity, while a bond represents debt owed by that entity to its creditors.

WITHOUT adapter (base model):
 ### Instruction:
What is the difference between a stock and a bond?

### Response:
A stock represents ownership in a company, while a bond represents debt owed by an investor to a company. Stocks are typically traded on stock exchanges, while bonds are not. Bonds have a fixed interest rate and maturity date, while stocks can be bought and sold at any time. Additionally, bonds may come with additional protections such as protection against inflation or default, which stocks do not typically offer. Finally,


In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# pull a sample of instructions from each domain's dataset to represent it
programming_samples = [ex['instruction'] for ex in programming_data.select(range(100))]
finance_samples = [ex['instruction'] for ex in finance_data.select(range(100))]

programming_embeddings = embedder.encode(programming_samples)
finance_embeddings = embedder.encode(finance_samples)

# centroid = average embedding for the domain
centroids = {
    "programming": np.mean(programming_embeddings, axis=0),
    "finance": np.mean(finance_embeddings, axis=0),
}

In [ ]:
def route_query(query, centroids, temperature=1.0):
    query_emb = embedder.encode(query)

    # cosine similarity to each centroid
    sims = {}
    for domain, centroid in centroids.items():
        sim = np.dot(query_emb, centroid) / (np.linalg.norm(query_emb) * np.linalg.norm(centroid))
        sims[domain] = sim

    # softmax over similarities to get weights
    sim_values = np.array(list(sims.values())) / temperature
    exp_sims = np.exp(sim_values - np.max(sim_values))  # numerical stability
    weights = exp_sims / exp_sims.sum()

    return dict(zip(sims.keys(), weights))

In [ ]:
print(route_query("How do I reverse a linked list in Python?", centroids, temperature=0.1))
print(route_query("What is the difference between a stock and a bond?", centroids, temperature=0.1))

{'programming': np.float32(0.933777), 'finance': np.float32(0.06622297)}
{'programming': np.float32(0.04328402), 'finance': np.float32(0.95671594)}


In [ ]:
print(route_query("How would you build a Python script to detect anomalies in stock trading data?", centroids, temperature=0.1))

{'programming': np.float32(0.8172818), 'finance': np.float32(0.18271823)}


In [ ]:
print(route_query("Explain how algorithmic trading systems use code to execute financial risk models.", centroids, temperature=0.1))
print(route_query("What programming techniques are used to detect fraud in banking transactions?", centroids, temperature=0.1))
print(route_query("How does encryption protect financial transaction data?", centroids, temperature=0.1))

{'programming': np.float32(0.2452593), 'finance': np.float32(0.7547407)}
{'programming': np.float32(0.48567736), 'finance': np.float32(0.51432264)}
{'programming': np.float32(0.026606549), 'finance': np.float32(0.9733934)}


In [ ]:
from peft import PeftModel

# reload a clean base model fresh — router combination needs both adapters loaded as named adapters
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

model = PeftModel.from_pretrained(model, "drive/MyDrive/LoRA-Adapters/adapters/programming_adapter", adapter_name="programming")
model.load_adapter("drive/MyDrive/LoRA-Adapters/adapters/finance_adapter", adapter_name="finance")

def generate_with_routing(query, centroids, model, tokenizer, temperature=0.1):
    weights = route_query(query, centroids, temperature=temperature)

    # combine adapters using router weights
    model.add_weighted_adapter(
        adapters=["programming", "finance"],
        weights=[weights["programming"], weights["finance"]],
        adapter_name="combined",
        combination_type="linear",
    )
    model.set_adapter("combined")

    prompt = f"### Instruction:\n{query}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    return tokenizer.decode(output[0], skip_special_tokens=True), weights

response, weights = generate_with_routing(
    "What programming techniques are used to detect fraud in banking transactions?",
    centroids, model, tokenizer
)
print(f"Router weights: {weights}\n")
print(response)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Router weights: {'programming': np.float32(0.48567736), 'finance': np.float32(0.51432264)}

### Instruction:
What programming techniques are used to detect fraud in banking transactions?

### Response:
Detecting fraud in banking transactions involves the use of machine learning algorithms. These algorithms analyze patterns and anomalies that may indicate fraudulent activity, such as unusual transaction amounts or frequent transactions from a specific location.


In [ ]:
model.set_adapter("programming")
inputs = tokenizer("### Instruction:\nWhat programming techniques are used to detect fraud in banking transactions?\n\n### Response:\n", return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Instruction:
What programming techniques are used to detect fraud in banking transactions?

### Response:
Detecting fraud in banking transactions involves several programming techniques. One common approach is using machine learning algorithms, such as decision trees or neural networks, to analyze patterns and anomalies in the data.

Another technique is using statistical methods, such as regression analysis or clustering, to identify potential fraudulent activities based on historical transaction data.

Additionally, some banks use artificial intelligence (AI) systems that can automatically flag suspicious activity by


In [ ]:
eval_queries = {
    "programming": [
        "How do you reverse a linked list in Python?",
        "What is the difference between a stack and a queue?",
        "Explain how recursion works with an example.",
        "What does Big O notation measure?",
    ],
    "finance": [
        "What is the difference between a stock and a bond?",
        "What does compound interest mean?",
        "Explain what a mutual fund is.",
        "What is the role of a central bank?",
    ],
    "mixed": [
        "What programming techniques are used to detect fraud in banking transactions?",
        "How would you build a trading bot that executes stock trades automatically?",
    ],
}

In [ ]:
import json

results = []

for domain, queries in eval_queries.items():
    for q in queries:
        prompt = f"### Instruction:\n{q}\n\n### Response:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # static adapter — use the "true" domain if known, else programming as default for mixed
        static_adapter = domain if domain in ["programming", "finance"] else "programming"
        model.set_adapter(static_adapter)
        with torch.no_grad():
            static_out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

        # router-weighted
        weights = route_query(q, centroids, temperature=0.1)
        model.add_weighted_adapter(
            adapters=["programming", "finance"],
            weights=[weights["programming"], weights["finance"]],
            adapter_name="combined_eval",
            combination_type="linear",
        )
        model.set_adapter("combined_eval")
        with torch.no_grad():
            router_out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        model.delete_adapter("combined_eval")  # clean up so adapter names don't pile up

        results.append({
            "query": q,
            "true_domain": domain,
            "router_weights": {k: float(v) for k, v in weights.items()},
            "static_response": tokenizer.decode(static_out[0], skip_special_tokens=True),
            "router_response": tokenizer.decode(router_out[0], skip_special_tokens=True),
        })

with open("drive/MyDrive/LoRA-Adapters/eval/my_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results.")

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1683: UserWarning: Adapter combined_eval was active which is now deleted. Setting active adapter to combined.
  warnings.warn(


Saved 10 results.


In [ ]:
import json

with open("drive/MyDrive/LoRA-Adapters/eval/my_eval_results.json") as f:
    results = json.load(f)

for r in results:
    print(f"[{r['true_domain']}] {r['query']}")
    print(f"  weights: {r['router_weights']}")
    print(f"  static:  {r['static_response'][-150:]}")  # last part of response, after the prompt echo
    print(f"  router:  {r['router_response'][-150:]}")
    print()

[programming] How do you reverse a linked list in Python?
  weights: {'programming': 0.9314224123954773, 'finance': 0.06857763230800629}
  static:  _list(head):
    prev = None
    current = head
    while current is not None:
        next_node = current.next  # Store the next node
        current
  router:  _list(head):
    prev = None
    current = head
    while current is not None:
        next_node = current.next  # Store the next node
        current

[programming] What is the difference between a stack and a queue?
  weights: {'programming': 0.23258261382579803, 'finance': 0.7674173712730408}
  static:   is also a linear data structure but it follows the First In, First Out (FIFO) principle. It allows elements to be added at the front of the queue and
  router:  hand, is a linear data structure that follows the First-In-First-Out (FIFO) principle, meaning elements are removed from the front of the queue first.

[programming] Explain how recursion works with an example.
  weights

In [ ]:
programming_samples = [ex['instruction'] for ex in programming_data]
finance_samples = [ex['instruction'] for ex in finance_data]

programming_embeddings = embedder.encode(programming_samples)
finance_embeddings = embedder.encode(finance_samples)

centroids = {
    "programming": np.mean(programming_embeddings, axis=0),
    "finance": np.mean(finance_embeddings, axis=0),
}

In [ ]:
print(route_query("What is the difference between a stack and a queue?", centroids, temperature=0.1))
print(route_query("What does Big O notation measure?", centroids, temperature=0.1))

{'programming': np.float32(0.25139016), 'finance': np.float32(0.74860984)}
{'programming': np.float32(0.11300678), 'finance': np.float32(0.8869932)}


In [ ]:
import json

results = []

for domain, queries in eval_queries.items():
    for q in queries:
        prompt = f"### Instruction:\n{q}\n\n### Response:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # static adapter — use the "true" domain if known, else programming as default for mixed
        static_adapter = domain if domain in ["programming", "finance"] else "programming"
        model.set_adapter(static_adapter)
        with torch.no_grad():
            static_out = model.generate(**inputs, max_new_tokens=80, do_sample=False)

        # router-weighted
        weights = route_query(q, centroids, temperature=0.1)
        model.add_weighted_adapter(
            adapters=["programming", "finance"],
            weights=[weights["programming"], weights["finance"]],
            adapter_name="combined_eval",
            combination_type="linear",
        )
        model.set_adapter("combined_eval")
        with torch.no_grad():
            router_out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        model.delete_adapter("combined_eval")  # clean up so adapter names don't pile up

        results.append({
            "query": q,
            "true_domain": domain,
            "router_weights": {k: float(v) for k, v in weights.items()},
            "static_response": tokenizer.decode(static_out[0], skip_special_tokens=True),
            "router_response": tokenizer.decode(router_out[0], skip_special_tokens=True),
        })

with open("drive/MyDrive/LoRA-Adapters/eval/my_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results.")

Saved 10 results.


In [ ]:
import json

with open("drive/MyDrive/LoRA-Adapters/eval/my_eval_results.json") as f:
    results = json.load(f)

for r in results:
    print(f"[{r['true_domain']}] {r['query']}")
    print(f"  weights: {r['router_weights']}")
    print(f"  static:  {r['static_response'][-150:]}")  # last part of response, after the prompt echo
    print(f"  router:  {r['router_response'][-150:]}")
    print()

[programming] How do you reverse a linked list in Python?
  weights: {'programming': 0.9367402195930481, 'finance': 0.06325973570346832}
  static:  _list(head):
    prev = None
    current = head
    while current is not None:
        next_node = current.next  # Store the next node
        current
  router:  _list(head):
    prev = None
    current = head
    while current is not None:
        next_node = current.next  # Store the next node
        current

[programming] What is the difference between a stack and a queue?
  weights: {'programming': 0.25139015913009644, 'finance': 0.7486098408699036}
  static:   is also a linear data structure but it follows the First In, First Out (FIFO) principle. It allows elements to be added at the front of the queue and
  router:  hand, is a linear data structure that follows the First-In-First-Out (FIFO) principle, meaning elements are removed from the front of the queue first.

[programming] Explain how recursion works with an example.
  weights